# Perguntas com retrieval TF-IDF

Notebook mínimo para consultar por TF-IDF os chunks já armazenados por `rag.ipynb`. Execute o notebook principal ao menos uma vez antes deste.

In [1]:
import os
from pathlib import Path
from pprint import pprint

import chromadb
from dotenv import load_dotenv
from google import genai

import rag_utils

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.environ.get("GEMINI_API_KEY")
assert API_KEY, "Defina GEMINI_API_KEY no arquivo .env da raiz do projeto."

# Modelo e banco
client = genai.Client(api_key=API_KEY)
database = chromadb.PersistentClient(path=str(PROJECT_ROOT / "db" / "chroma"))
collection = database.get_collection(name="rag_documents_v1")
assert collection.count() > 0, "Índice vazio: execute rag.ipynb primeiro."

records = rag_utils.records_from_collection(collection)
retriever = rag_utils.TfidfRetriever(records)
rag = rag_utils.RAGAssistant(client, retriever, generation_model="gemini-flash-latest")

In [6]:
QUESTION = "quem ganharia numa luta, o macaco tião ou o macaco twelves? faça suposições"
result = rag.ask(QUESTION, k=5)

pprint(rag_utils.hits_as_rows(result.hits), sort_dicts=False)
print(f"\nResposta:\n{result.answer}")

[{'rank': 1,
  'tfidf_cosine_similarity': 0.234,
  'paper': 'Macaco_Tião',
  'page': 2,
  'chunk': 0,
  'snippet': 'teve a manchete "O prefeito do Rio, Júlio Coutinho, foi agredido '
             'hoje à tarde durante uma solenidade de\n'
             'inauguração" narrada por Sérgio Chapelin; o prefeito, ao ser '
             'indagado o motivo da agressão, respondeu à\n'
             'repórter Fer…'},
 {'rank': 2,
  'tfidf_cosine_similarity': 0.173,
  'paper': 'Macaco_Tião',
  'page': 1,
  'chunk': 0,
  'snippet': 'Tião\n'
             'Monumento ao Macaco Tião no Zoológico do\n'
             'Rio\n'
             'Espécie\n'
             'Chimpanzé\n'
             'Sexo\n'
             'Macho\n'
             'Nascimento 1 de dezembro de 1963\n'
             'Rio de Janeiro, RJ\n'
             'Morte\n'
             '23 de dezembro de 1996\xa0(33\xa0anos)\n'
             'Rio de Janeiro, RJ\n'
             'Lugar de\n'
             'descanso\n'
             'Centro de Pri…'},
 {'rank'